In [186]:
# Зареждане на набора от данни
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root="data/Cora", name="Cora")
data = dataset[0]

In [198]:
# Дефиниране на Graph Transformer модела
import torch
import torch.nn.functional as F
from torch_geometric.nn import TransformerConv

class GraphTransformer(torch.nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = TransformerConv(
            dataset.num_features,
            32,
            heads=2,
            dropout=0.5
        )

        self.conv2 = TransformerConv(
            32 * 2,
            dataset.num_classes,
            heads=1,
            concat=False,
            dropout=0.5
        )

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x,
                      p=0.5,
                      training=self.training)

        x = self.conv2(x, edge_index)

        return x

In [199]:
# Създаване на модела
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = GraphTransformer().to(device)

data = data.to(device)

    weight_decay=5e-4)

In [ ]:
# Инициализиране на оптимизатора
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,

In [200]:
# Обучение
import time
start = time.time()

for epoch in range(200):

    model.train()

    optimizer.zero_grad()

    out = model(
        data.x,
        data.edge_index
    )

    loss = F.cross_entropy(
        out[data.train_mask],
        data.y[data.train_mask]
    )

    loss.backward()

    optimizer.step()

training_time = time.time() - start

In [201]:
# Оценяване
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
model.eval()

with torch.no_grad():

    out = model(
        data.x,
        data.edge_index
    )

pred = out.argmax(dim=1)

y_true = data.y[data.test_mask].cpu()

y_pred = pred[data.test_mask].cpu()

accuracy = accuracy_score(
    y_true,
    y_pred
)

f1 = f1_score(
    y_true,
    y_pred,
    average="macro"
)

num_params = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro F1-score: {f1:.4f}")
print(f"Параметри: {num_params}")
print(f"Време: {training_time:.2f} s")

Accuracy: 0.7850
Macro F1-score: 0.7734
Параметри: 368924
Време: 10.38 s
